# Reranker Fine-tune PoC — val_009 only

**Hypothesis**: feature-enriched LoRA fine-tuning of Qwen3-Reranker-8B on Swiss-legal
in-domain triplets will deliver a clear R@K lift on val_009 (the hardest val query —
v5.x got 0/14 gold, v7 got 6/10 reachable gold). Note that only 10 of the 14 gold
documents survive Stage-A retrieval into the top-2000 candidate pool we rerank here.

**Why val_009 alone is enough**: if the approach works for the worst query, it works
for the rest. If it doesn't, the full-scale training won't either.

**Feature set** (mirrors what a human Swiss lawyer uses), embedded *inside* the
official Qwen3-Reranker `<Query>` / `<Document>` template the base model was trained
on (see Phase 2):

| Feature | Lawyer's check |
|---|---|
| `target_codes` / `code` | "Right code? ZGB for family, BGG for appeals?" |
| `target_areas` / `area` | "Right legal area?" |
| `target_topics` / `topic` | "Right legal topic?" |
| `target_concepts` / `concepts` | "Right key concepts?" |
| `role` | "Foundational or procedural? Definition vs duty vs scope?" |
| `chapter_neighbors` | "Near other articles I'd expect to cite?" |
| `popularity` | "Cited often in Swiss case law?" |
| `text` | "Does the rule actually answer the query?" |

**Implementation notes** (aligned with the official Qwen3-Reranker + vLLM docs):
- Prompts use the **official Qwen3-Reranker template** (`<Instruct>: ... <Query>: ... <Document>: ...`
  inside the standard system prefix and `<think>` suffix) so the base model's reranker
  prior transfers; enriched metadata is embedded *inside* `<Query>` and `<Document>`.
- Scoring uses **vLLM's native `runner="pooling"`** with `hf_overrides` converting
  `lm_head` to a 2-class `Qwen3ForSequenceClassification` head — the documented
  efficient path (returns a 0-1 relevance score in one forward pass instead of
  generating a token + extracting logprobs over the 151k-vocab).
- LoRA targets attention + MLP projections only (lm_head untouched), so the merged
  model is still compatible with vLLM's score API.

**Inputs required on Drive**:
- `/MyDrive/swiss_law/research/reranker_train/reranker_train_enriched_30k.parquet`  (upload — 12.3 MB)
- `/MyDrive/swiss_law/research/reranker_train/val009_candidates_enriched.parquet`  (upload — 0.7 MB)

**Decision criteria** (printed at the end):
- Relative lift on R@K_gold ≥ 50% → CONCLUSIVE YES (proceed to full-scale)
- Lift 20-50%                   → PROMISING (try harder negatives)
- Lift < 20%                    → CONCLUSIVE NO (re-architect)


## Phase 0 — Setup

In [ ]:
import os, sys, subprocess, json, time, gc, io, re, math
from pathlib import Path
sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding="utf-8", errors="replace")

IS_COLAB = "google.colab" in sys.modules
print(f"Colab: {IS_COLAB}")
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    # vLLM >=0.9.2 is required for native Qwen3-Reranker support
    # (runner="pooling" with Qwen3ForSequenceClassification hf_overrides).
    # unsloth + trl are pulled in up front so Phase 4 doesn't pip-install mid-run.
    subprocess.run(["pip","install","-q","-U",
        "vllm>=0.9.2",
        "transformers>=4.51.0","peft>=0.13.0","accelerate>=1.0.0",
        "datasets>=3.0.0","trl>=0.12.0","bitsandbytes>=0.44.0",
        "unsloth","unsloth_zoo",
        "pandas==2.2.3","pyarrow==16.1.0","numpy==1.26.4","tqdm"], check=True)
print("Setup OK")


## Phase 1 — Paths + load val_009 query + enriched candidates

In [ ]:
DRIVE_ROOT     = Path("/content/drive/MyDrive/swiss_law")
VAL_CSV        = DRIVE_ROOT / "data"     / "val.csv"
VAL_ASPECTS    = DRIVE_ROOT / "research" / "concept_embedding_path" / "multi_aspect" / "val_aspects.parquet"
TRAIN_PARQ     = DRIVE_ROOT / "research" / "reranker_train" / "reranker_train_enriched_30k.parquet"
VAL009_PARQ    = DRIVE_ROOT / "research" / "reranker_train" / "val009_candidates_enriched.parquet"

OUT_DIR        = DRIVE_ROOT / "research" / "reranker_finetune_poc_val009"
OUT_DIR.mkdir(parents=True, exist_ok=True)
LORA_DIR       = OUT_DIR / "lora_adapter"
LORA_DIR.mkdir(parents=True, exist_ok=True)
MERGED_DIR     = OUT_DIR / "qwen3_reranker_8b_swiss_legal_merged"

QID            = "val_009"
MODEL_NAME     = "Qwen/Qwen3-Reranker-8B"
# Two separate caps — explicit, never silently overridden later:
#   scoring (vLLM): wider, accommodates full enriched candidate text
#   training (Unsloth): narrower to keep VRAM bounded
MAX_LEN_SCORE  = 1536
MAX_LEN_TRAIN  = 1024
LR             = 1e-4
BATCH_SIZE     = 8
GRAD_ACCUM     = 2          # effective batch 16
EPOCHS         = 1
LORA_RANK      = 16
LORA_ALPHA     = 32

# Task instruction injected into <Instruct>: in the Qwen3-Reranker template.
# Mirrors the role of the default "Given a web search query, retrieve relevant
# passages..." instruction, but anchored to the Swiss-legal citation task.
SCORE_INSTRUCTION = (
    "Given a Swiss legal question, decide whether a competent Swiss lawyer "
    "writing a legal opinion on the Query would cite the Document. Consider "
    "matching legal codes, legal areas, legal topics, and key concepts."
)

print(f"Verifying inputs:")
for p in [VAL_CSV, VAL_ASPECTS, TRAIN_PARQ, VAL009_PARQ]:
    print(f"  {'OK' if p.exists() else 'MISSING'}  {p}")

import pandas as pd

val_df = pd.read_csv(VAL_CSV)
query_text = str(val_df[val_df.query_id == QID].iloc[0]["query"])
print(f"\n=== val_009 question ({len(query_text)} chars) ===\n{query_text[:600]}...")

asp_df = pd.read_parquet(VAL_ASPECTS)
v009_row = asp_df[asp_df.query_id == QID].iloc[0]
aspects = list(v009_row["aspects"])
print(f"\n=== val_009 aspects ({len(aspects)}) ===")
for a in aspects:
    print(f"  {a.get('id')}: {a.get('label')}  (weight={float(a.get('weight',0)):.2f})")

# Derive query metadata - handle numpy arrays from parquet
def to_list(v):
    if v is None: return []
    try: return list(v)
    except TypeError: return []

expected_codes = to_list(v009_row.get("expected_codes"))
target_topics  = [a.get("label","") for a in aspects if a.get("label")]
target_concepts = []
for a in aspects:
    for c in to_list(a.get("concepts_en")):
        target_concepts.append(c)
target_concepts = list(dict.fromkeys(target_concepts))[:8]

target_areas = []
area_keywords = {
    "maintenance":    "civil law",
    "child":          "civil law",
    "marriage":       "civil law",
    "divorce":        "civil law",
    "property":       "civil law",
    "enforcement":    "civil law / debt enforcement",
    "appeal":         "federal supreme court procedure",
    "detention":      "criminal procedure",
    "criminal":       "criminal procedure",
    "tax":            "tax law",
    "labor":          "labor law",
    "employment":     "labor law",
    "capitalization": "civil law",
}
for a in aspects:
    lbl = (a.get("label","") or "").lower()
    for k, area in area_keywords.items():
        if k in lbl:
            target_areas.append(area); break
target_areas = list(dict.fromkeys(target_areas))

query_meta = {
    "target_codes":    ", ".join(expected_codes) if expected_codes else "(any)",
    "target_areas":    ", ".join(target_areas) if target_areas else "(any)",
    "target_topics":   "; ".join(target_topics)[:300],
    "target_concepts": ", ".join(target_concepts)[:300],
}
print()
print("=== val_009 derived query metadata ===")
for k, v in query_meta.items():
    print(f"  {k}: {v}")

# Load enriched candidates
val_cands = pd.read_parquet(VAL009_PARQ)
print(f"\nLoaded {len(val_cands)} val_009 candidates  "
      f"(gold_in_topk = {int(val_cands.is_gold.sum())} of {len(val_cands)})")

train_df = pd.read_parquet(TRAIN_PARQ)
print(f"Loaded {len(train_df):,} training triplets")


## Phase 2 — Define Qwen3-Reranker prompt format (official template + enriched fields)

We use the **official Qwen3-Reranker chat template** (system prefix + `<Instruct>` /
`<Query>` / `<Document>` + `<think>` suffix), the exact format the base model was
trained on. Enriched lawyer-style metadata is embedded *inside* the `<Query>` and
`<Document>` sections — this preserves the base model's reranker prior while still
exposing every feature the lawyer table in the header lists.


In [ ]:
# Official Qwen3-Reranker template (from model card / vLLM example):
#   prefix       = "<|im_start|>system\nJudge whether the Document meets..."
#   user content = "<Instruct>: {instruction}\n<Query>: {query}\n<Document>: {doc}"
#   suffix       = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
# We keep this verbatim and inject enriched lawyer-features inside the Query and
# Document strings so the base model's prior carries over.

QWEN3_RR_PREFIX = (
    "<|im_start|>system\n"
    "Judge whether the Document meets the requirements based on the Query and "
    "the Instruct provided. Note that the answer can only be \"yes\" or \"no\"."
    "<|im_end|>\n<|im_start|>user\n"
)
QWEN3_RR_SUFFIX = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"

def render_query_content(text, target_codes, target_areas, target_topics, target_concepts):
    """Body of <Query>: — keeps the original question plus the structured aspects."""
    return (
        f"{text[:400]}\n"
        f"target_codes: {target_codes} | target_areas: {target_areas} | "
        f"target_topics: {target_topics} | target_concepts: {target_concepts}"
    )

def render_document_content(citation, code, area, role, title, topic, concepts,
                            chapter_neighbors, popularity_bucket, text):
    """Body of <Document>: — leads with the citation tuple, then text."""
    return (
        f"citation: {citation} | code: {code} | area: {area} | role: {role}\n"
        f"title: {title[:120]}\n"
        f"topic: {topic[:120]}\n"
        f"concepts: {concepts[:200]}\n"
        f"chapter_neighbors: {chapter_neighbors[:200]}\n"
        f"popularity: {popularity_bucket}\n"
        f"text: {text[:700]}"
    )

def render_query_for_score(query_text, query_meta, instruction=SCORE_INSTRUCTION):
    """Full query string passed to llm.score() — prefix + <Instruct> + <Query>:."""
    q_body = render_query_content(
        query_text,
        query_meta["target_codes"], query_meta["target_areas"],
        query_meta["target_topics"], query_meta["target_concepts"],
    )
    return f"{QWEN3_RR_PREFIX}<Instruct>: {instruction}\n<Query>: {q_body}\n"

def render_document_for_score(candidate_fields):
    """Full document string passed to llm.score() — <Document>: ... + suffix."""
    d_body = render_document_content(**candidate_fields)
    return f"<Document>: {d_body}{QWEN3_RR_SUFFIX}"

def format_full_prompt_for_training(query_text, query_meta, candidate_fields,
                                    instruction=SCORE_INSTRUCTION):
    """Concatenate query + document into a single training prompt
    (ends right before the yes/no answer token)."""
    return (
        render_query_for_score(query_text, query_meta, instruction)
        + render_document_for_score(candidate_fields)
    )

def bucketize(n):
    n = int(n)
    if n >= 1000: return "very_high"
    if n >= 100:  return "high"
    if n >= 10:   return "medium"
    return "low"

# Sanity-check render on a known-gold candidate
sample_row = val_cands[val_cands.is_gold].iloc[0]
sample_fields = {
    "citation":          sample_row["citation"],
    "code":              sample_row["code"],
    "area":              sample_row["area"] or "",
    "role":              sample_row["role"] or "(unknown)",
    "title":             sample_row.get("title", "") or "",
    "topic":             sample_row.get("topic", "") or "",
    "concepts":          sample_row.get("concepts", "") or "",
    "chapter_neighbors": sample_row.get("chapter_neighbors", "") or "(none)",
    "popularity_bucket": sample_row["popularity_bucket"],
    "text":              sample_row["text"],
}
sample_prompt = format_full_prompt_for_training(query_text, query_meta, sample_fields)
print(f"=== Sample full prompt for a GOLD val_009 candidate ({sample_row['citation']}) ===")
print(f"Total length: {len(sample_prompt)} chars")
print("-" * 80)
print(sample_prompt)


## Phase 3 — BASELINE: score val_009 candidates with off-the-shelf Qwen3-Reranker-8B

Uses vLLM's native `runner="pooling"` path (the documented efficient route): the model
loads as `Qwen3ForSequenceClassification`, `lm_head` is collapsed to a 2-vector
(no / yes) classifier on the fly, and `llm.score()` returns a 0-1 relevance score
per (query, document) pair in one forward pass.


In [ ]:
from vllm import LLM
from transformers import AutoTokenizer

print(f"Loading {MODEL_NAME} (baseline, no LoRA) via vLLM `runner=pooling` ...")
# Tokenizer kept around for the training cell; not strictly needed by score API.
qwen3_tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, padding_side="left")

# Per vLLM official example (runner="pooling") (examples/pooling/score/qwen3_reranker.py):
# `hf_overrides` re-routes the model to Qwen3ForSequenceClassification at load
# time, extracts the no/yes rows from lm_head, and converts them to a single-vec
# binary classifier. `is_original_qwen3_reranker=True` enables the softmax(yes,no)
# conversion. Result: `outputs.score` is a 0-1 relevance score, no token decoding.
HF_OVERRIDES_RERANK = {
    "architectures":              ["Qwen3ForSequenceClassification"],
    "classifier_from_token":      ["no", "yes"],
    "is_original_qwen3_reranker": True,
}

_orig_stdout, _orig_stderr = sys.stdout, sys.stderr
sys.stdout = sys.__stdout__; sys.stderr = sys.__stderr__
try:
    qwen3_llm = LLM(
        model=MODEL_NAME,
        runner="pooling",
        dtype="bfloat16",
        max_model_len=MAX_LEN_SCORE,
        gpu_memory_utilization=0.85,
        enforce_eager=False,
        trust_remote_code=True,
        hf_overrides=HF_OVERRIDES_RERANK,
    )
finally:
    sys.stdout, sys.stderr = _orig_stdout, _orig_stderr
print(f"Loaded.")

# Build the single query string (same for all candidates) and the N document strings.
query_str = render_query_for_score(query_text, query_meta)
def build_doc_str(row):
    return render_document_for_score({
        "citation":          row.citation,
        "code":              row.code,
        "area":              row.area or "",
        "role":              row.role or "(unknown)",
        "title":             row.title or "",
        "topic":             row.topic or "",
        "concepts":          row.concepts or "",
        "chapter_neighbors": row.chapter_neighbors or "(none)",
        "popularity_bucket": row.popularity_bucket,
        "text":              row.text or "",
    })

print(f"\nBuilding {len(val_cands)} candidate documents ...")
doc_strs = [build_doc_str(r) for r in val_cands.itertuples()]
print(f"  mean doc length: {sum(len(d) for d in doc_strs)//len(doc_strs)} chars; "
      f"query length: {len(query_str)} chars")

def _extract_score(o):
    """vLLM's ScoringRequestOutput.outputs is `ScoringOutput` (single, old API)
    or `list[ScoringOutput]` (newer API used by Qwen3-Reranker examples)."""
    out = o.outputs
    if isinstance(out, list):
        return float(out[0].score)
    return float(out.score)

print(f"\nBaseline scoring (vLLM score API, 1-vs-N) ...")
t0 = time.time()
# llm.score(single_query, list_of_documents) pairs the query against each doc.
score_outs = qwen3_llm.score(query_str, doc_strs)
baseline_scores = [_extract_score(o) for o in score_outs]
print(f"Done in {(time.time()-t0)/60:.1f} min  ({len(baseline_scores)} scores)")
val_cands["baseline_score"] = baseline_scores

# Diagnostic: gold vs non-gold mean (now in 0-1 probability space, not logit-diff)
gold_mean = val_cands[val_cands.is_gold].baseline_score.mean()
non_mean  = val_cands[~val_cands.is_gold].baseline_score.mean()
print(f"\nBaseline gold mean:     {gold_mean:.4f}  (n={int(val_cands.is_gold.sum())})")
print(f"Baseline non-gold mean: {non_mean:.4f}  (n={int((~val_cands.is_gold).sum())})")
print(f"Separation:             {gold_mean-non_mean:+.4f}")

# Free vLLM before training
del qwen3_llm
gc.collect()
import torch
torch.cuda.empty_cache()
print(f"\nGPU mem after free: {torch.cuda.memory_allocated()/1024**3:.1f} GB")


## Phase 4 — LoRA fine-tune on 30k enriched triplets

In [ ]:
# === Phase 4 — Unsloth LoRA fine-tune (RTX 6000 Blackwell optimized) ===
# unsloth + trl were installed in Phase 0, so no mid-notebook pip install here.

try: del model
except NameError: pass
import gc, torch, os, time
gc.collect(); torch.cuda.empty_cache()
print(f"GPU mem before: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

os.environ["TORCH_CUDA_ARCH_LIST"] = "12.0"
# Reset any cached unsloth module state from a previous run in the same kernel.
import sys as _sys
for _mod in list(_sys.modules):
    if _mod.startswith(("unsloth", "bitsandbytes")):
        del _sys.modules[_mod]

print(f"Config: BATCH={BATCH_SIZE}, GRAD_ACCUM={GRAD_ACCUM} (eff={BATCH_SIZE*GRAD_ACCUM}), "
      f"MAX_LEN_TRAIN={MAX_LEN_TRAIN}")

from unsloth import FastLanguageModel

print(f"\nLoading {MODEL_NAME} via Unsloth ...")
t0 = time.time()
model, qwen3_tok = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_LEN_TRAIN,
    dtype           = torch.bfloat16,
    load_in_4bit    = False,
    load_in_8bit    = False,
    full_finetuning = False,
)
print(f"  loaded in {time.time()-t0:.1f}s  ({torch.cuda.memory_allocated()/1024**3:.1f} GB)")

yes_id = qwen3_tok("yes", add_special_tokens=False).input_ids[0]
no_id  = qwen3_tok("no",  add_special_tokens=False).input_ids[0]
print(f"  yes_id={yes_id}, no_id={no_id}")

# LoRA targets attention + MLP projections (lm_head intentionally left untouched
# so the merged model stays compatible with vLLM's runner="pooling" path, which
# extracts the no/yes rows from lm_head).
model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_RANK,
    target_modules             = ["q_proj","k_proj","v_proj","o_proj",
                                  "gate_proj","up_proj","down_proj"],
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = 0.05,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 42,
    use_rslora                 = False,
    loftq_config               = None,
)
model.print_trainable_parameters()
print(f"GPU mem after LoRA setup: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

class EnrichedSFTDataset(Dataset):
    """SFT dataset that emits Qwen3-Reranker-formatted prompts with a single
    yes/no answer token. Loss is masked to ONLY that final token, matching
    how the base reranker decides — a clean binary classification objective."""
    def __init__(self, df, tok, max_len=MAX_LEN_TRAIN):
        self.examples = []
        for r in df.itertuples():
            q_meta = {
                "target_codes":    str(r.target_codes or "(any)"),
                "target_areas":    str(r.target_areas or "(any)"),
                "target_topics":   str(r.target_topics or "")[:300],
                "target_concepts": str(r.target_concepts or "")[:300],
            }
            for side, is_pos in [("pos", True), ("neg", False)]:
                cand_fields = {
                    "citation":          str(getattr(r, f"{side}_citation")),
                    "code":              str(getattr(r, f"{side}_code")),
                    "area":              str(getattr(r, f"{side}_area") or ""),
                    "role":              str(getattr(r, f"{side}_role") or "(unknown)"),
                    "title":             str(getattr(r, f"{side}_title") or ""),
                    "topic":             str(getattr(r, f"{side}_topic") or ""),
                    "concepts":          str(getattr(r, f"{side}_concepts") or ""),
                    "chapter_neighbors": str(getattr(r, f"{side}_chapter_neighbors") or "(none)"),
                    "popularity_bucket": bucketize(getattr(r, f"{side}_popularity")),
                    "text":              str(getattr(r, f"{side}_text") or "")[:600],
                }
                self.examples.append((r.query, q_meta, cand_fields, is_pos))
        self.tok = tok
        self.max_len = max_len

    def __len__(self): return len(self.examples)

    def __getitem__(self, idx):
        q, q_meta, cand, is_pos = self.examples[idx]
        target = "yes" if is_pos else "no"
        full = format_full_prompt_for_training(q, q_meta, cand) + target
        enc = self.tok(full, return_tensors="pt", add_special_tokens=False,
                       max_length=self.max_len, truncation=True)
        ids = enc.input_ids[0]
        labels = torch.full_like(ids, -100)
        labels[-1] = ids[-1]   # supervise ONLY the final yes/no token
        return {"input_ids": ids, "labels": labels}

def collate(batch):
    max_len = max(b["input_ids"].size(0) for b in batch)
    pad_id = qwen3_tok.pad_token_id or qwen3_tok.eos_token_id
    input_ids = torch.full((len(batch), max_len), pad_id, dtype=torch.long)
    labels    = torch.full((len(batch), max_len), -100, dtype=torch.long)
    attention_mask = torch.zeros((len(batch), max_len), dtype=torch.long)
    for i, b in enumerate(batch):
        L = b["input_ids"].size(0)
        input_ids[i, -L:] = b["input_ids"]
        labels[i, -L:]    = b["labels"]
        attention_mask[i, -L:] = 1
    return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}

ds = EnrichedSFTDataset(train_df, qwen3_tok, max_len=MAX_LEN_TRAIN)
dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate,
                num_workers=2, pin_memory=True)
print(f"\nDataset: {len(ds):,} examples, dataloader: {len(dl):,} batches per epoch")

# Sanity-check ONE example: the last token of a positive example must == yes_id,
# negative must == no_id. Catches any tokenizer-merge surprises before training.
for i, (_, _, _, is_pos) in enumerate(ds.examples[:4]):
    item = ds[i]
    last = int(item["input_ids"][-1].item())
    expected = yes_id if is_pos else no_id
    assert last == expected, f"Example {i}: last token {last} != expected {expected}"
print("  last-token label sanity-check passed.")

total_steps = len(dl) * EPOCHS // GRAD_ACCUM
optim = AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
sched = get_linear_schedule_with_warmup(optim, num_warmup_steps=int(0.03*total_steps),
                                        num_training_steps=total_steps)

print(f"\nTraining: {total_steps:,} optimizer steps over {EPOCHS} epoch(s)")
print(f"  Effective batch={BATCH_SIZE*GRAD_ACCUM}, LR={LR}, max_len={MAX_LEN_TRAIN}")
print(f"  GPU mem at training start: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

t_start = time.time()
running_loss = 0.0
step = 0
optim.zero_grad()
for epoch in range(EPOCHS):
    for i, batch in enumerate(dl):
        batch = {k: v.cuda(non_blocking=True) for k, v in batch.items()}
        out = model(**batch)
        loss = out.loss / GRAD_ACCUM
        loss.backward()
        running_loss += out.loss.item()
        if (i + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            optim.step(); sched.step(); optim.zero_grad()
            step += 1
            if step % 50 == 0:
                avg = running_loss / (GRAD_ACCUM * 50)
                running_loss = 0.0
                elapsed = (time.time()-t_start) / 60
                eta = elapsed / step * (total_steps - step)
                peak = torch.cuda.max_memory_allocated()/1024**3
                print(f"  step {step:>5}/{total_steps}  loss={avg:.4f}  "
                      f"lr={sched.get_last_lr()[0]:.2e}  elapsed={elapsed:.1f}min  "
                      f"eta={eta:.1f}min  peak_mem={peak:.1f}GB")
print(f"\nTraining done in {(time.time()-t_start)/60:.1f} min")

model.save_pretrained(str(LORA_DIR))
qwen3_tok.save_pretrained(str(LORA_DIR))
print(f"LoRA adapter -> {LORA_DIR}")

del model, optim, sched, dl, ds
gc.collect()
torch.cuda.empty_cache()
print(f"GPU mem after free: {torch.cuda.memory_allocated()/1024**3:.1f} GB")


## Phase 5 — Merge LoRA + re-score val_009 candidates

We merge the LoRA adapter into the base weights and re-load via vLLM's `runner="pooling"`
path with the same `hf_overrides`. Because LoRA only touched attention + MLP
projections (not `lm_head`), the no/yes classifier extraction still works on the
merged checkpoint and we get a direct apples-to-apples comparison against the
baseline scores from Phase 3.


In [ ]:
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

print(f"Merging LoRA into base ...")
t0 = time.time()
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16, device_map="cpu", trust_remote_code=True
)
merged = PeftModel.from_pretrained(base, str(LORA_DIR))
merged = merged.merge_and_unload()
merged.save_pretrained(str(MERGED_DIR), safe_serialization=True)
qwen3_tok.save_pretrained(str(MERGED_DIR))
print(f"Merged -> {MERGED_DIR}  in {time.time()-t0:.1f}s")

del base, merged
gc.collect()
torch.cuda.empty_cache()

print(f"\nLoading merged model via vLLM `runner=pooling` ...")
_orig_stdout, _orig_stderr = sys.stdout, sys.stderr
sys.stdout = sys.__stdout__; sys.stderr = sys.__stderr__
try:
    qwen3_llm_ft = LLM(
        model=str(MERGED_DIR),
        runner="pooling",
        dtype="bfloat16",
        max_model_len=MAX_LEN_SCORE,
        gpu_memory_utilization=0.85,
        enforce_eager=False,
        trust_remote_code=True,
        hf_overrides=HF_OVERRIDES_RERANK,
    )
finally:
    sys.stdout, sys.stderr = _orig_stdout, _orig_stderr
print(f"Loaded.")

print(f"\nFine-tuned scoring on the same {len(doc_strs)} val_009 pairs ...")
t0 = time.time()
ft_score_outs = qwen3_llm_ft.score(query_str, doc_strs)
ft_scores = [_extract_score(o) for o in ft_score_outs]
print(f"Done in {(time.time()-t0)/60:.1f} min")
val_cands["ft_score"] = ft_scores

# Diagnostic
gold_mean_ft = val_cands[val_cands.is_gold].ft_score.mean()
non_mean_ft  = val_cands[~val_cands.is_gold].ft_score.mean()
gold_mean_base = val_cands[val_cands.is_gold].baseline_score.mean()
non_mean_base  = val_cands[~val_cands.is_gold].baseline_score.mean()

print(f"\n=== Score separation gold vs non-gold (0-1 probability space) ===")
print(f"  Baseline:    gold={gold_mean_base:.4f}   non-gold={non_mean_base:.4f}   sep={gold_mean_base-non_mean_base:+.4f}")
print(f"  Fine-tuned:  gold={gold_mean_ft:.4f}   non-gold={non_mean_ft:.4f}   sep={gold_mean_ft-non_mean_ft:+.4f}")

del qwen3_llm_ft
gc.collect()
torch.cuda.empty_cache()


## Phase 6 — VERDICT: R@K side-by-side + per-gold rank + F1@K_gold + decision


In [ ]:
def recall_at_k(df, score_col, k):
    top = df.sort_values(score_col, ascending=False).head(k)
    return int(top.is_gold.sum()) / max(1, int(df.is_gold.sum()))

K_LIST = [10, 25, 50, 100, 200, 500]
GOLD_TOTAL = 14  # full val_009 gold count (from gold_doc_sets)

print("="*80)
print(f"  RERANKER FINE-TUNE — val_009 (Stage A top-2000, 10 gold in pool)")
print("="*80)

print(f"\n{'K':<8}{'baseline R':>14}{'fine-tuned R':>14}{'delta':>10}{'rel %':>10}")
for K in K_LIST:
    base_r = recall_at_k(val_cands, "baseline_score", K)
    ft_r   = recall_at_k(val_cands, "ft_score", K)
    delta = ft_r - base_r
    rel = (delta / max(1e-6, base_r)) * 100
    print(f"R@{K:<6}{base_r:>14.3f}{ft_r:>14.3f}{delta:>+10.3f}{rel:>+9.1f}%")

# R@K_gold (= 14 — full gold count): the most relevant operating point
base_rkg = recall_at_k(val_cands, "baseline_score", GOLD_TOTAL)
ft_rkg   = recall_at_k(val_cands, "ft_score", GOLD_TOTAL)
delta_rkg = ft_rkg - base_rkg
rel_rkg = (delta_rkg / max(1e-6, base_rkg)) * 100
print(f"R@K_gold={GOLD_TOTAL:<3}{base_rkg:>14.3f}{ft_rkg:>14.3f}{delta_rkg:>+10.3f}{rel_rkg:>+9.1f}%")

# Per-gold ranking comparison
print(f"\n=== Per-gold ranking (where each gold sits in the sorted list) ===")
val_sorted_base = val_cands.sort_values("baseline_score", ascending=False).reset_index(drop=True)
val_sorted_ft   = val_cands.sort_values("ft_score",       ascending=False).reset_index(drop=True)
print(f"  {'citation':<25}  {'baseline rank':>14}  {'fine-tuned rank':>16}  {'change':>10}")
gold_dids = set(val_cands[val_cands.is_gold].did)
for did in gold_dids:
    cit = val_cands[val_cands.did == did].iloc[0]["citation"]
    rank_base = int(val_sorted_base[val_sorted_base.did == did].index[0]) + 1
    rank_ft   = int(val_sorted_ft[val_sorted_ft.did == did].index[0]) + 1
    change = rank_base - rank_ft   # positive = improved
    arrow = "+" if change > 0 else ""
    print(f"  {cit:<25}  {rank_base:>14}  {rank_ft:>16}  {arrow}{change:>+10}")

# F1 at K = K_gold = 14
def f1(p, r): return 0.0 if (p+r)==0 else 2*p*r/(p+r)
top_base = val_sorted_base.head(GOLD_TOTAL)
top_ft   = val_sorted_ft.head(GOLD_TOTAL)
correct_base = int(top_base.is_gold.sum())
correct_ft   = int(top_ft.is_gold.sum())
p_base = correct_base / GOLD_TOTAL
p_ft   = correct_ft / GOLD_TOTAL
r_base = correct_base / GOLD_TOTAL
r_ft   = correct_ft / GOLD_TOTAL
f1_base = f1(p_base, r_base)
f1_ft   = f1(p_ft, r_ft)

print(f"\n=== F1 at K_gold ({GOLD_TOTAL}) ===")
print(f"  Baseline:    P={p_base:.3f}  R={r_base:.3f}  F1={f1_base:.3f}  (correct={correct_base}/{GOLD_TOTAL})")
print(f"  Fine-tuned:  P={p_ft:.3f}  R={r_ft:.3f}  F1={f1_ft:.3f}  (correct={correct_ft}/{GOLD_TOTAL})")
print(f"  Delta F1:    {f1_ft - f1_base:+.3f}")

print(f"\n" + "="*80)
print(f"  VERDICT")
print(f"="*80)
if rel_rkg >= 50.0:
    print(f"  CONCLUSIVE YES — R@{GOLD_TOTAL} relative lift {rel_rkg:+.1f}% >= 50%.")
    print(f"  Feature-enriched in-domain fine-tuning works on Swiss legal.")
    print(f"  Full-scale training pipeline (5M edges + synthetic queries) is justified.")
elif rel_rkg >= 20.0:
    print(f"  PROMISING — lift {rel_rkg:+.1f}% in [20%, 50%).")
    print(f"  Approach works but needs harder negatives + more diverse query synthesis.")
    print(f"  Worth scaling, but expect 0.45-0.55 F1 plateau without more work.")
else:
    print(f"  CONCLUSIVE NO — lift {rel_rkg:+.1f}% < 20%.")
    print(f"  Either: (a) training data doesn't transfer to val's question style,")
    print(f"          (b) the model can't learn from these features,")
    print(f"          (c) the eval is dominated by candidates the LLM cannot distinguish.")
    print(f"  Recommend: diagnose training loss curve + sample predictions before re-architecting.")


## Phase 7 — Save outputs

In [ ]:
val_cands.to_parquet(OUT_DIR / "val009_scores_baseline_vs_ft.parquet", index=False)
with open(OUT_DIR / "verdict.json", "w") as f:
    json.dump({
        "qid": QID,
        "gold_total": GOLD_TOTAL,
        "gold_in_topk": int(val_cands.is_gold.sum()),
        "baseline": {"R_at_K_gold": base_rkg, "F1_at_K_gold": f1_base,
                     "P": p_base, "correct": correct_base},
        "fine_tuned": {"R_at_K_gold": ft_rkg, "F1_at_K_gold": f1_ft,
                       "P": p_ft, "correct": correct_ft},
        "relative_lift_pct_R_at_K_gold": rel_rkg,
        "delta_F1": f1_ft - f1_base,
        "training_config": {
            "model": MODEL_NAME, "rank": LORA_RANK, "alpha": LORA_ALPHA,
            "epochs": EPOCHS, "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM,
            "lr": LR, "max_len_score": MAX_LEN_SCORE,
            "max_len_train": MAX_LEN_TRAIN,
        },
    }, f, indent=2)
print(f"Saved to {OUT_DIR}")
